# SQL CONNECTION


In [12]:
# 1. Load and clean
loan_train = pd.read_csv('./Data/train.csv')

clean_age = loan_train.loc[loan_train['person_age'] <= 100, 'person_age'].median()
clean_emp = loan_train.loc[loan_train['person_emp_length'] <= 60, 'person_emp_length'].median()
loan_train.loc[loan_train['person_age'] > 100, 'person_age'] = clean_age
loan_train.loc[loan_train['person_emp_length'] > 60, 'person_emp_length'] = clean_emp

In [9]:
import sqlite3
import pandas as pd

In [10]:
conn = sqlite3.connect('credit_risk.db')


In [13]:
loan_train.to_sql('loans', conn, if_exists='replace', index=False)

58645

In [14]:
check = pd.read_sql("SELECT COUNT(*) as row_count FROM loans", conn)
print(check)

   row_count
0      58645


In [ ]:
# ============================================
# QUERY 1: Overall default rate
# ============================================
query1 = """
SELECT 
    loan_status,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM loans), 2) as percentage
FROM loans
GROUP BY loan_status
"""
print("--- Overall default rate ---")
print(pd.read_sql(query1, conn))


# ============================================
# QUERY 2: Default rate by loan grade
# ============================================
query2 = """
SELECT 
    loan_grade,
    COUNT(*) as total_loans,
    SUM(loan_status) as defaults,
    ROUND(AVG(loan_status) * 100, 2) as default_rate_pct
FROM loans
GROUP BY loan_grade
ORDER BY default_rate_pct DESC
"""
print("\n--- Default rate by loan grade ---")
print(pd.read_sql(query2, conn))


# ============================================
# QUERY 3: Default rate by home ownership
# ============================================
query3 = """
SELECT 
    person_home_ownership,
    COUNT(*) as total_loans,
    SUM(loan_status) as defaults,
    ROUND(AVG(loan_status) * 100, 2) as default_rate_pct
FROM loans
GROUP BY person_home_ownership
ORDER BY default_rate_pct DESC
"""
print("\n--- Default rate by home ownership ---")
print(pd.read_sql(query3, conn))


# ============================================
# QUERY 4: Default rate by loan intent
# ============================================
query4 = """
SELECT 
    loan_intent,
    COUNT(*) as total_loans,
    SUM(loan_status) as defaults,
    ROUND(AVG(loan_status) * 100, 2) as default_rate_pct
FROM loans
GROUP BY loan_intent
ORDER BY default_rate_pct DESC
"""
print("\n--- Default rate by loan intent ---")
print(pd.read_sql(query4, conn))


# ============================================
# QUERY 5: Average interest rate and loan amount, defaulted vs not
# ============================================
query5 = """
SELECT 
    loan_status,
    ROUND(AVG(loan_int_rate), 2) as avg_interest_rate,
    ROUND(AVG(loan_amnt), 2) as avg_loan_amount,
    ROUND(AVG(person_income), 2) as avg_income
FROM loans
GROUP BY loan_status
"""
print("\n--- Averages by loan status ---")
print(pd.read_sql(query5, conn))


# ============================================
# QUERY 6: A more advanced one - default rate by income bracket AND home ownership
# ============================================
query6 = """
SELECT 
    CASE 
        WHEN person_income < 30000 THEN 'Low Income'
        WHEN person_income BETWEEN 30000 AND 70000 THEN 'Mid Income'
        ELSE 'High Income'
    END as income_bracket,
    person_home_ownership,
    COUNT(*) as total_loans,
    ROUND(AVG(loan_status) * 100, 2) as default_rate_pct
FROM loans
GROUP BY income_bracket, person_home_ownership
ORDER BY income_bracket, default_rate_pct DESC
"""
print("\n--- Default rate by income bracket AND home ownership (combined) ---")
print(pd.read_sql(query6, conn))


# ============================================
# QUERY 7: Highest-risk segment
# ============================================
query7 = """
SELECT 
    loan_grade,
    person_home_ownership,
    COUNT(*) as total_loans,
    ROUND(AVG(loan_status) * 100, 2) as default_rate_pct
FROM loans
GROUP BY loan_grade, person_home_ownership
HAVING COUNT(*) >= 50
ORDER BY default_rate_pct DESC
LIMIT 10
"""
print("\n--- Top 10 riskiest grade+ownership combinations (min 50 loans) ---")
print(pd.read_sql(query7, conn))

--- Overall default rate ---
   loan_status  count  percentage
0            0  50295       85.76
1            1   8350       14.24

--- Default rate by loan grade ---
  loan_grade  total_loans  defaults  default_rate_pct
0          G           33        27             81.82
1          E         1009       631             62.54
2          F          149        91             61.07
3          D         5034      2988             59.36
4          C        11036      1494             13.54
5          B        20400      2087             10.23
6          A        20984      1032              4.92

--- Default rate by home ownership ---
  person_home_ownership  total_loans  defaults  default_rate_pct
0                  RENT        30594      6809             22.26
1                 OTHER           89        15             16.85
2              MORTGAGE        24824      1483              5.97
3                   OWN         3138        43              1.37

--- Default rate by loan intent ---

In [16]:
# Save query 6 results
income_ownership_risk = pd.read_sql(query6, conn)
income_ownership_risk.to_csv('risk_by_income_and_ownership.csv', index=False)
print(f"Saved {len(income_ownership_risk)} rows to risk_by_income_and_ownership.csv")

# Save query 7 results
top_risk_segments = pd.read_sql(query7, conn)
top_risk_segments.to_csv('top_10_riskiest_segments.csv', index=False)
print(f"Saved {len(top_risk_segments)} rows to top_10_riskiest_segments.csv")

Saved 12 rows to risk_by_income_and_ownership.csv
Saved 10 rows to top_10_riskiest_segments.csv
